In [ ]:
from __future__ import annotations

import json
import re
from pathlib import Path
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd

# -----------------------------
# Exchange on/off switches
# -----------------------------
EXCHANGES = {
    "cox": True,   # Oslo Stock Exchange
    "hex": True,   # Nasdaq Helsinki
    "isx": True,   # Nasdaq Iceland
    "obx": True,   # Nasdaq Copenhagen
    "stx": True,   # Nasdaq Stockholm
}

# -----------------------------
# Paths
# -----------------------------
BASE_DIR = Path(".").resolve()   # notebook is inside PROF
DATA_DIR = BASE_DIR.parent / "data"
OUT_DIR  = BASE_DIR / "prof_components_extracted"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Per-exchange input and mapping dirs (derived automatically from exchange key)
def get_exchange_dirs(exchange_key: str):
    input_dir   = DATA_DIR / f"{exchange_key}_xlsx"
    mapping_dir = BASE_DIR / "mappings" / f"mappings_{exchange_key}"
    return input_dir, mapping_dir

# Your sheets are consistent: statement begins at Excel row 19 => skip first 18 rows
SKIPROWS = 17

# Only keep years <= 2024
MAX_YEAR = 2024

# Variable logic
PRIORITY_VARS = {"REVT", "XRD", "BE", "MIB"}
SUM_VARS = {"COGS", "XSGA_COMPONENTS", "XINT"}

ALL_VARS = ["REVT", "COGS", "XSGA_COMPONENTS", "XRD", "XINT", "BE", "MIB"]

In [ ]:
# -----------------------------
# Helpers
# -----------------------------
def parse_year(colname: str) -> Optional[int]:
    """
    Extract year from column name.
    Handles headers like '2024', 'FY2024', '31-12-2024', '2024-12-31', etc.
    Returns int year or None.
    """
    if colname is None:
        return None
    s = str(colname).strip()
    # Find a 4-digit year 19xx/20xx
    m = re.search(r"(19\d{2}|20\d{2})", s)
    if not m:
        return None
    return int(m.group(1))


def to_float(x):
    """Robust numeric parsing incl. European formats and thousand separators."""
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return np.nan
    if isinstance(x, (int, float, np.number)):
        return float(x)

    s = str(x).strip()
    if s == "" or s.lower() in {"nan", "none"}:
        return np.nan

    # remove spaces / NBSP
    s = s.replace("\u00a0", "").replace(" ", "")

    # if both comma and dot exist, assume comma is thousands sep
    if "," in s and "." in s:
        s = s.replace(",", "")
    else:
        # if only comma exists, treat comma as decimal separator
        if "," in s and "." not in s:
            s = s.replace(",", ".")

    try:
        return float(s)
    except ValueError:
        return np.nan



def detect_header_row(xlsx_path: Path, sheet_name, max_scan_rows: int = 80) -> Optional[int]:
    """
    Returns 0-indexed header row where column A equals 'Field Name'.
    Reads only column A to avoid out-of-bounds issues.
    """
    scan = pd.read_excel(
        xlsx_path,
        sheet_name=sheet_name,
        header=None,
        nrows=max_scan_rows,
        usecols="A",    
        engine="openpyxl",
    )

    for i in range(len(scan)):
        a0 = scan.iloc[i, 0]
        a0 = "" if pd.isna(a0) else str(a0).strip()
        if a0.lower() == "field name":
            return i

    return None


def read_sheet_all_years(xlsx_path: Path, sheet_name: str) -> pd.DataFrame:
    """
    Reads the sheet using the detected 'Field Name' row as the header.
    Returns a DataFrame with column 0 as labels and remaining columns as years/dates.
    """
    header_row = detect_header_row(xlsx_path, sheet_name)
    n = 0
    if header_row is None:
        # fallback: try your old behavior
        print("NO HEADER ROW DETECTED, FALLBACK TO SKIPROWS=18")
        n += 1
        df = pd.read_excel(xlsx_path, sheet_name=sheet_name, skiprows=18)
    else:
        df = pd.read_excel(xlsx_path, sheet_name=sheet_name, header=header_row)

    if df.empty or df.shape[1] == 0:
        return df

    label_col = df.columns[0]
    df[label_col] = df[label_col].astype(str).str.strip()
    df = df[df[label_col] != ""].copy()
    return df, n


def keep_year_cols_upto(df: pd.DataFrame, max_year: int) -> Tuple[pd.DataFrame, List[int]]:
    """
    Keeps only the year columns with parsed year <= max_year.
    Returns (df_filtered, years_sorted_desc_or_asc).
    """
    if df.empty:
        return df, []

    label_col = df.columns[0]
    year_cols = []
    years = []
    for c in df.columns[1:]:
        y = parse_year(c)
        if y is not None and y <= max_year:
            year_cols.append(c)
            years.append(y)

    keep_cols = [label_col] + year_cols
    df2 = df[keep_cols].copy()

    # Standardize columns to year ints (may create duplicates)
    new_cols = [label_col] + [parse_year(c) for c in year_cols]
    df2.columns = new_cols

    # If duplicate year columns exist, collapse by taking first non-null across them (per row)
    if len(set(new_cols[1:])) != len(new_cols[1:]):
        label = df2.columns[0]
        out = df2[[label]].copy()
        for y in sorted(set(new_cols[1:])):
            cols_y = [c for c in df2.columns[1:] if c == y]
            block = df2[cols_y].applymap(to_float)
            out[y] = block.bfill(axis=1).iloc[:, 0]
        df2 = out

    years_sorted = sorted([c for c in df2.columns[1:] if isinstance(c, int)])
    return df2, years_sorted


def extract_label_series_with_duplicates(df_years: pd.DataFrame, row_label: str) -> pd.Series:
    """
    Extract a time series for a given row_label, handling duplicates:
    if multiple rows have same label, for each year take first non-null from top-most row.
    """
    if df_years.empty:
        return pd.Series(dtype=float)

    label_col = df_years.columns[0]
    hits = df_years[df_years[label_col] == row_label]
    if hits.empty:
        return pd.Series(dtype=float)

    block = hits.drop(columns=[label_col]).applymap(to_float)
    picked = block.bfill(axis=0).iloc[0]
    picked.name = row_label
    return picked


def priority_across_labels(series_list: List[pd.Series]) -> pd.Series:
    """Given a list of label-series in priority order, pick first non-null per year."""
    if not series_list:
        return pd.Series(dtype=float)
    df = pd.concat(series_list, axis=1)
    out = df.bfill(axis=1).iloc[:, 0]
    return out


def sum_across_labels(series_list: List[pd.Series]) -> pd.Series:
    """Sum series across labels (align on years)."""
    if not series_list:
        return pd.Series(dtype=float)
    df = pd.concat(series_list, axis=1)
    return df.sum(axis=1, min_count=1)


def load_mapping(mapping_path: Path) -> Dict:
    return json.loads(mapping_path.read_text(encoding="utf-8"))

def get_company_metadata(xlsx_path: Path) -> Tuple[str, str]:
    """
    Reads company name from cell B2 and TRBC Industry Group from cell B5.
    Returns (company_name_without_ticker, industry).
    """
    meta = pd.read_excel(xlsx_path, sheet_name=0, header=None, usecols="A:B", nrows=6)

    raw_name = "" if pd.isna(meta.iat[1, 1]) else str(meta.iat[1, 1]).strip()
    industry  = "" if pd.isna(meta.iat[4, 1]) else str(meta.iat[4, 1]).strip()

    company_name = re.sub(r"\s*\([^)]*\)\s*$", "", raw_name).strip()
    return company_name, industry

def trbc_industry_to_sector(industry: str) -> str:
    mapping = {
        "Oil & Gas": "Energy",
        "Oil & Gas Related Equipment and Services": "Energy",
        "Renewable Energy": "Energy",
        "Freight & Logistics Services": "Industrials",
        "Machinery, Tools, Heavy Vehicles, Trains & Ships": "Industrials",
        "Construction & Engineering": "Industrials",
        "Professional & Commercial Services": "Industrials",
        "Aerospace & Defense": "Industrials",
        "Passenger Transportation Services": "Industrials",
        "Food & Tobacco": "Consumer Staples",
        "Specialty Retailers": "Consumer Discretionary",
        "Diversified Retail": "Consumer Discretionary",
        "Automobiles & Auto Parts": "Consumer Discretionary",
        "Hotels & Entertainment Services": "Consumer Discretionary",
        "Biotechnology & Medical Research": "Health Care",
        "Pharmaceuticals": "Health Care",
        "Healthcare Equipment & Supplies": "Health Care",
        "Banking Services": "Financials",
        "Investment Banking & Investment Services": "Financials",
        "Investment Holding Companies": "Financials",
        "Insurance": "Financials",
        "Metals & Mining": "Materials",
        "Chemicals": "Materials",
        "Paper & Forest Products": "Materials",
        "Containers & Packaging": "Materials",
        "Software & IT Services": "Information Technology",
        "Semiconductors & Semiconductor Equipment": "Information Technology",
        "Electronic Equipment & Parts": "Information Technology",
        "Computers, Phones & Household Electronics": "Information Technology",
        "Communications & Networking": "Communication Services",
        "Media & Publishing": "Communication Services",
        "Telecommunications Services": "Communication Services",
        "Electric Utilities & IPPs": "Utilities",
        "Real Estate Operations": "Real Estate",
    }
    if industry is None:
        return "Other"
    return mapping.get(str(industry).strip(), "Other")

In [ ]:
# -----------------------------
# Main extraction per firm
# -----------------------------
def extract_firm_components(xlsx_path: Path, mapping_path: Path, max_year: int = 2024) -> pd.DataFrame:
    mapping = load_mapping(mapping_path)
    variables = mapping.get("variables", [])

    sheet_cache: Dict[str, pd.DataFrame] = {}
    years_union = set()
    extracted: Dict[str, pd.Series] = {}

    for var_item in variables:
        var = var_item.get("variable", "")
        if var not in ALL_VARS:
            continue

        choices = var_item.get("final_choice", [])

        if not choices:
            extracted[var] = pd.Series(dtype=float)
            continue

        label_series_list: List[pd.Series] = []

        for ch in choices:
            sheet = ch["sheet_name"]
            label = ch["row_label"]

            if sheet not in sheet_cache:
                df_raw, n = read_sheet_all_years(xlsx_path, sheet)
                df_years, _ = keep_year_cols_upto(df_raw, max_year)
                sheet_cache[sheet] = df_years

            s = extract_label_series_with_duplicates(sheet_cache[sheet], label)
            if not s.empty:
                label_series_list.append(s)

        if var in PRIORITY_VARS:
            s_var = priority_across_labels(label_series_list)
        elif var in SUM_VARS:
            s_var = sum_across_labels(label_series_list)
        else:
            s_var = sum_across_labels(label_series_list)

        extracted[var] = s_var
        years_union.update(list(s_var.index))

    years_sorted = sorted([y for y in years_union if isinstance(y, int)])
    df_out = pd.DataFrame(index=years_sorted)
    df_out.index.name = "Year"

    for var in ALL_VARS:
        s = extracted.get(var, pd.Series(dtype=float))
        df_out[var] = s.reindex(years_sorted)

    df_out = df_out.fillna(0.0)
    return df_out, n

In [ ]:
# -----------------------------
# Run extraction for all active exchanges
# -----------------------------
active_exchanges = [key for key, enabled in EXCHANGES.items() if enabled]
print(f"Active exchanges: {active_exchanges}")

total_ok, total_skip = 0, 0
last_n = 0

for exchange in active_exchanges:
    input_dir, mappings_dir = get_exchange_dirs(exchange)

    if not mappings_dir.exists():
        print(f"[{exchange}] Mappings dir not found: {mappings_dir}, skipping.")
        continue
    if not input_dir.exists():
        print(f"[{exchange}] Input dir not found: {input_dir}, skipping.")
        continue

    mapping_files = sorted(mappings_dir.glob("*.json"))
    print(f"\n[{exchange}] Found {len(mapping_files)} mapping files")

    n_ok, n_skip = 0, 0

    for mp in mapping_files:
        firm_id = mp.stem
        xlsx_path = input_dir / f"{firm_id}.xlsx"
        if not xlsx_path.exists():
            n_skip += 1
            continue

        df, last_n = extract_firm_components(xlsx_path, mp, max_year=MAX_YEAR)

        df = df.reset_index()
        company_name, industry = get_company_metadata(xlsx_path)
        sector = trbc_industry_to_sector(industry)

        df.insert(1, "Ticker",      firm_id)
        df.insert(2, "Exchange",    exchange.upper())  # tag which exchange
        df.insert(3, "CompanyName", company_name)
        df.insert(4, "Industry",    industry)
        df.insert(5, "Sector",      sector)

        out_path = OUT_DIR / f"{firm_id}.csv"
        df.to_csv(out_path, index=False)
        n_ok += 1

    print(f"[{exchange}] Extracted {n_ok} firms. Skipped {n_skip} missing workbooks.")
    total_ok += n_ok
    total_skip += n_skip

print(f"\nTotal: {total_ok} firms extracted, {total_skip} skipped.")
print(f"Header fallbacks used in {last_n} sheets.")

In [ ]:
BASE_DIR = Path(".").resolve()
IN_DIR = BASE_DIR / "prof_components_extracted"

for fp in sorted(IN_DIR.glob("*.csv")):
    df = pd.read_csv(fp, index_col=0)

    # Expense-like items -> positive
    for c in ["REVT", "COGS", "XSGA_COMPONENTS", "XRD", "XINT"]:
        if c in df.columns:
            df[c] = df[c].abs()

    numer = df["REVT"] - df["COGS"] - (df["XSGA_COMPONENTS"] - df["XRD"]) - df["XINT"]
    denom = (df["BE"] + df["MIB"]).replace({0.0: np.nan})

    df["PROF"] = numer / denom
    df.to_csv(fp)

In [ ]:
# ── Accruals extraction config ────────────────────────────────────────────────
ACC_OUT_DIR = BASE_DIR / "acc_components_extracted"

ACC_PRIORITY_VARS = {"ACT", "CHE", "LCT", "TXP", "AT", "OANCF"}  
ACC_SUM_VARS      = {"STD", "PPEGT"}              
ACC_ALL_VARS      = ["ACT", "CHE", "LCT", "STD", "TXP", "PPEGT", "AT", "OANCF"]

def extract_firm_acc_components(xlsx_path: Path, mapping_path: Path, max_year: int = 2024):
    """Same logic as extract_firm_components but targets accruals variables."""
    mapping   = load_mapping(mapping_path)
    variables = mapping.get("variables", [])

    sheet_cache: Dict[str, pd.DataFrame] = {}
    years_union = set()
    extracted: Dict[str, pd.Series] = {}
    n_fallbacks = 0

    for var_item in variables:
        var = var_item.get("variable", "")
        if var not in ACC_ALL_VARS:
            continue

        choices = var_item.get("final_choice", [])
        if not choices:
            extracted[var] = pd.Series(dtype=float)
            continue

        label_series_list: List[pd.Series] = []
        for ch in choices:
            sheet = ch["sheet_name"]
            label = ch["row_label"]

            if sheet not in sheet_cache:
                df_raw, n = read_sheet_all_years(xlsx_path, sheet)
                n_fallbacks += n
                df_years, _ = keep_year_cols_upto(df_raw, max_year)
                sheet_cache[sheet] = df_years

            s = extract_label_series_with_duplicates(sheet_cache[sheet], label)
            if not s.empty:
                label_series_list.append(s)

        if var in ACC_SUM_VARS:
            s_var = sum_across_labels(label_series_list)
        else:
            s_var = priority_across_labels(label_series_list)

        extracted[var] = s_var
        years_union.update(list(s_var.index))

    years_sorted = sorted([y for y in years_union if isinstance(y, int)])
    df_out = pd.DataFrame(index=years_sorted)
    df_out.index.name = "Year"

    for var in ACC_ALL_VARS:
        s = extracted.get(var, pd.Series(dtype=float))
        df_out[var] = s.reindex(years_sorted)

    return df_out, n_fallbacks

In [ ]:
# ── Run accruals extraction for all active exchanges ──────────────────────────
ACC_OUT_DIR = BASE_DIR / "acc_components_extracted"
ACC_OUT_DIR.mkdir(parents=True, exist_ok=True)

active_exchanges = [key for key, enabled in EXCHANGES.items() if enabled]
print(f"Active exchanges: {active_exchanges}")

total_ok, total_skip, total_fallbacks = 0, 0, 0

for exchange in active_exchanges:
    input_dir, _ = get_exchange_dirs(exchange)
    acc_mappings_dir = BASE_DIR / "mappings" / f"mappings_{exchange}"  # adjust if acc mappings are elsewhere
    # If your accruals mappings are in a separate folder, update the line below:
    # acc_mappings_dir = BASE_DIR.parent / "accruals" / "acc_mappings" / f"acc_mappings_{exchange}"

    if not acc_mappings_dir.exists():
        print(f"[{exchange}] Accruals mappings dir not found: {acc_mappings_dir}, skipping.")
        continue
    if not input_dir.exists():
        print(f"[{exchange}] Input dir not found: {input_dir}, skipping.")
        continue

    mapping_files = sorted(acc_mappings_dir.glob("*.json"))
    xlsx_files    = {f.stem for f in input_dir.glob("*.xlsx")}

    missing = [mp.stem for mp in mapping_files if mp.stem not in xlsx_files]
    if missing:
        print(f"[{exchange}] {len(missing)} mappings with no matching xlsx")

    print(f"\n[{exchange}] Found {len(mapping_files)} accruals mapping files")
    n_ok, n_skip, n_fb = 0, 0, 0

    for mp in mapping_files:
        firm_id   = mp.stem
        xlsx_path = input_dir / f"{firm_id}.xlsx"

        if not xlsx_path.exists():
            n_skip += 1
            continue

        df, fb = extract_firm_acc_components(xlsx_path, mp, max_year=MAX_YEAR)
        n_fb += fb

        df = df.reset_index()
        company_name, industry = get_company_metadata(xlsx_path)
        sector = trbc_industry_to_sector(industry)

        df.insert(1, "Ticker",      firm_id)
        df.insert(2, "Exchange",    exchange.upper())
        df.insert(3, "CompanyName", company_name)
        df.insert(4, "Industry",    industry)
        df.insert(5, "Sector",      sector)

        out_path = ACC_OUT_DIR / f"{firm_id}.csv"
        df.to_csv(out_path, index=False)
        n_ok += 1

    print(f"[{exchange}] Extracted {n_ok} firms. Skipped {n_skip}. Fallbacks: {n_fb}.")
    total_ok += n_ok
    total_skip += n_skip
    total_fallbacks += n_fb

print(f"\nTotal: {total_ok} firms extracted, {total_skip} skipped, {total_fallbacks} fallbacks.")